In [1]:
import dustpy
import dustpy.constants as c
import astropy.constants as apc
import numpy as np
import matplotlib.pyplot as plt

import _opacity
import _Temperature 

c_light = apc.c.cgs.value
k_B = apc.k_B.cgs.value
sigma_sb = apc.sigma_sb.cgs.value




A newer version of DustPy is available.
This version:   1.0.8
Latest version: 1.0.9

Upgrade with
pip install dustpy --upgrade



In [2]:
###initialisation of the dustpy simulation (necessary because class Temperature is using values from there)###

sim = dustpy.Simulation()

#define the turbulent parameter alpha
sim.ini.gas.alpha = 1e-2

#initialize simulation framework
sim.initialize()

# if you want to use fixed mixing parameters instead of the same as sim.ini.gas.alpha, the delta parameters can be modified here:
#sim.dust.delta.rad = 0.001
#sim.dust.delta.turb = 0.001
#sim.dust.delta.vert = 0.001

sim.writer.overwrite = True
sim.gas.boundary.inner.value = 0
sim.dust.boundary.inner.value = 0

In [3]:
###function for using the half time-step for updating###

old_preparator = sim.integrator.preparator

sim.Sigma_gas_old = sim.gas.Sigma.copy()
sim.Sigma_dust_old = sim.dust.Sigma.copy()
def remember_old_sigma(sim):
    sim.Sigma_gas_old = sim.gas.Sigma.copy()
    sim.Sigma_dust_old = sim.dust.Sigma.copy()
    old_preparator.beat(sim)

sim.integrator.preparator = remember_old_sigma
sim.Sigma_gas_old = sim.gas.Sigma.copy()


In [4]:
###class opacity is called###

opac_file = 'default_opacities_smooth.npz'
b = np.load(opac_file)
print(b.files)
opac = _opacity.opacity(sim, opac_file)

['a', 'lam', 'k_abs', 'k_sca', 'g', 'rho_s']


In [5]:
###class Temperature is called to be used as an updater for sim.gas.T.updater###

temp_model = _Temperature.Temperature(opac)

T = temp_model.T_dustpy(sim)
sim.gas.T.updater.updater = temp_model.T_dustpy
sim.update()
T = np.asarray(sim.gas.T.data)
print(f"calculated temperatures: {T}")

calculated temperatures: [217.49291969 210.10922624 202.97620269 196.08533903 189.42841412
 182.99748597 176.78488217 170.78319082 164.98525161 159.38414733
 153.97319563 148.74594098 143.69614703 138.81778914 134.10504722
 129.55229875 125.15411208 120.90523998 116.80061336 112.8353352
 109.00467475 105.30406186 101.72908152  98.27546862  94.93910285
  91.71600375  88.60232604  85.59435496  82.68850184  79.88129989
  77.16939997  74.54956667  72.01867441  69.57370372  67.21173763
  64.92995822  62.72564321  60.59616276  58.5389763   56.55162951
  54.63175139  52.77705145  50.98531695  49.25441025  47.58226632
  45.9668902   44.40635468  42.89879797  41.44242149  40.03548771
  38.67631809  37.36329109  36.0948402   34.8694521   33.68566485
  32.54206614  31.4372916   30.37002318  29.33898759  28.34295475
  27.38073635  26.45118442  25.55318996  24.68568162  23.84762443
  23.03801854  22.25589806  21.50032988  20.77041258  20.06527533
  19.38407687  18.7260045   18.09027311  17.47612424

In [ ]:
###starting the DustPy simulation with the defined parameters and the updated temperature###
sim.run() 


DustPy v1.0.8

Documentation: https://stammler.github.io/dustpy/
PyPI:          https://pypi.org/project/dustpy/
GitHub:        https://github.com/stammler/dustpy/

Please cite Stammler & Birnstiel (2022).

Checking for mass conservation...

    - Sticking:
        max. rel. error:  2.82e-14
        for particle collision
            m[114] =  1.93e+04 g    with
            m[116] =  3.73e+04 g
    - Full fragmentation:
        max. rel. error:  5.55e-16
        for particle collision
            m[113] =  1.39e+04 g    with
            m[119] =  1.00e+05 g
    - Erosion:
        max. rel. error:  1.78e-15
        for particle collision
            m[110] =  5.18e+03 g    with
            m[118] =  7.20e+04 g

Writing file data/data0000.hdf5
Writing dump file data/frame.dmp
Writing file data/data0001.hdf5
Writing dump file data/frame.dmp
Writing file data/data0002.hdf5
Writing dump file data/frame.dmp
Writing file data/data0003.hdf5
Writing dump file data/frame.dmp
Writing file data/d

In [ ]:
###plotting the default and the updated temperatures for personal reference###

op = np.load("mean_opacities.npz")
Ttab = op["T"]
kapP = op["kappaP"]
kapR = op["kappaR"]
sigma_sb = apc.sigma_sb.cgs.value
gamma = 7/5
r = sim.grid.r
r_au = np.asarray(sim.grid.r) / c.au
Sigma = sim.gas.Sigma
H = sim.gas.Hp
R_star = sim.star.R
T_star = sim.star.T
kappaRg = 1e-3
kappaPg = 1e-3

exp = int(np.log10(sim.ini.gas.alpha))


h = H / r
Theta0 = 2 * 4 * h / 7
R_rim = 1
Theta = Theta0 + 0.5 * (1 - Theta0) * (1- np.tanh((r - R_rim)/0.1))


Sigmad_old = sim.Sigma_dust_old
Sigmag_old = sim.Sigma_gas_old
Sigmad_new = sim.dust.Sigma
Sigmag_new = sim.gas.Sigma
Sigmad = 0.5 * (Sigmad_new + Sigmad_old)
Sigmag = 0.5 * (Sigmag_new + Sigmag_old)
Sigma_dust_tot = Sigmad.sum(axis=1)
Sigma_dust_tot = np.maximum(Sigma_dust_tot, 1e-10)
#tau_R = 0.5 * Sigma_dust_tot * kapR 
#tau_P = 0.5 * Sigma_dust_tot * kapP 
tau_R = 0.5 * Sigma_dust_tot * kapR + 0.5 * Sigmag * kappaRg
tau_P = 0.5 * Sigma_dust_tot * kapP + 0.5 * Sigmag * kappaPg
tau_eff = (3 * tau_R / 8) + np.sqrt(3) / 4 + 1 / (4 * tau_P)

nu = sim.gas.alpha * sim.gas.cs**2 / sim.grid.OmegaK
Mdot = 3 * np.pi * Sigma * nu

# reference temperatures from: C. Dullemond, 2013, Theoretical Models of the Structure of Protoplanetary Disks, Les Houches
T_eff_accr = (3/(8 * np.pi * sigma_sb) * Mdot * sim.grid.OmegaK**2)**0.25 * (3/4 * tau_eff)**0.25
T_eff_accr_surf = (3/(8 * np.pi * sigma_sb) * Mdot * sim.grid.OmegaK**2)**0.25
T_eff_irr = (0.05 * sim.star.L/(4*np.pi*sigma_sb*r**2))**0.25

# reference from: E. I. Chiang and P. Goldreich. Spectral Energy Distributions of T Tauri Stars with Passive Circumstellar Disks. 
# The Astrophysical Journal, 490(1):368–376, Nov. 1997. doi: 10.1086/304869.
CG97 =  (Theta / 4)**0.25 * (R_star/r)**0.5 * T_star





fig, ax3 = plt.subplots(1, 1, dpi=300)

ax3.plot(r_au, T, color = 'green', label = fr'updated ($\alpha = 10^{{{exp}}}$)')

# feel free to activate the reference plots if needed
#ax3.plot(r_au, T0, color = 'red',label = 'DustPy default')
#ax3.plot(r_au, T_eff_accr,'--', c='orange', alpha=0.5, zorder=0, label = r'$T^{accr}_{eff}$')
#ax3.plot(r_au, T_eff_accr_surf,'--', c='grey', alpha=0.5, zorder=0, label = r'$T^{accr}_{eff,surf}$')
#ax3.plot(r_au, T_eff_irr, '--', c='purple', alpha=0.3, zorder=0, label = r'$T^{irr}_{eff}$')
#ax3.plot(r_au, CG97,ls=':', c='red', alpha=0.5, zorder=0, label='CG97')

ax3.set_xscale('log')
ax3.set_xlim(1, 900)
ax3.set_ylim(1,3000)
ax3.set_yscale('log')
ax3.grid('True', which = 'both')
ax3.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax3.set_ylabel(r'$T\,\left[\mathrm{K}\right]$')
ax3.set_title(r'temperature after $10^{5}$years')

plt.legend(loc = 'best')

plt.show()

In [ ]:
# function for plotting the single heating-terms embedded in _Temperature.py
temp_model.plot_heating()
# function for plotting the opacities embedded in _opacity.py in 2d and 3d
opac.plot_kappa()


In [ ]:
###default DustPy ipanel plots###

dustpy.plot.ipanel("data", it=10)
dustpy.plot.ipanel("data", it=15)
dustpy.plot.ipanel("data", it=21)